In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable
import logging
from datetime import datetime


In [0]:
# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

In [0]:
# Configuration
SOURCE_PATH = "abfss://raw@collibri.dfs.core.windows.net/landing/"
TARGET_TABLE = "interviews_dev.bronze.turbine_raw"
FILE_PATTERN = "data_group_*.csv"


In [0]:
def read_csv_files(source_path):
    """Read CSV files - uses existing spark variable"""
    logger.info(f"Reading CSV files from {source_path}...")
    
    df = (
        spark.read
            .format("csv")
            .option("header", "true")
            .option("inferSchema", "true")
            .option("cloudFiles.inferColumnTypes", "true")
            .option("nullValue", "")
            .option("emptyValue", None)
            .option("rescuedDataColumn", "_rescued_data")
            .load(f"{source_path}{FILE_PATTERN}")
            .withColumn("_source_file", F.col("_metadata.file_path"))
            .withColumn("_file_name", F.col("_metadata.file_name"))
            .withColumn(
                "_file_group",
                F.regexp_extract(F.col("_metadata.file_name"), r"data_group_(\d+)\.csv", 1)
            )
            .withColumn("_ingested_at", F.current_timestamp())
            .withColumn("_ingestion_date", F.current_date())
    )
    
    row_count = df.count()
    logger.info(f"Read {row_count:,} rows")
    df.show(5, truncate=False)
    
    return df


def deduplicate_data(df):
    """Deduplicate by turbine_id + timestamp"""
    logger.info("Deduplicating data...")
    
    original_count = df.count()
    
    dedupe_window = Window.partitionBy("turbine_id", "timestamp").orderBy(
        F.col("_ingested_at").desc()
    )
    
    deduped = df.withColumn("_row_number", F.row_number().over(dedupe_window)) \
                .filter(F.col("_row_number") == 1) \
                .drop("_row_number")
    
    logger.info(f"Original: {original_count:,}, After dedup: {deduped.count():,}")
    
    return deduped

def merge_into_bronze(source_df, target_table):
    """MERGE into Delta table"""
    logger.info(f"Merging into {target_table}...")
    
    table_exists = spark.catalog.tableExists(target_table)
    
    if not table_exists:
        logger.info(f"Creating new table: {target_table}")
        source_df.write.format("delta").mode("overwrite").saveAsTable(target_table)
        return "created"
    
    existing_count = spark.table(target_table).count()
    logger.info(f"Existing records: {existing_count:,}")
    
    target_delta = DeltaTable.forName(spark, target_table)
    
    (
        target_delta.alias("target")
        .merge(
            source_df.alias("source"),
            "target.turbine_id = source.turbine_id AND target.timestamp = source.timestamp"
        )
        .whenMatchedUpdate(set={
            "power_output": "source.power_output",
            "wind_speed": "source.wind_speed",
            "wind_direction": "source.wind_direction",
            "_rescued_data": "source._rescued_data",
            "_source_file": "source._source_file",
            "_file_name": "source._file_name",
            "_file_group": "source._file_group",
            "_ingested_at": "source._ingested_at"
        })
        .whenNotMatchedInsertAll()
        .execute()
    )
    
    new_count = spark.table(target_table).count()
    logger.info(f"Total after merge: {new_count:,}")
    
    return "merged"

def calculate_data_quality_metrics(target_table):
    """Calculate data quality metrics"""
    logger.info("Calculating data quality metrics...")
    
    stats = spark.sql(f"""
        SELECT COUNT(*) AS total_rows,
               COUNT(DISTINCT turbine_id) AS unique_turbines,
               MIN(timestamp) AS min_timestamp,
               MAX(timestamp) AS max_timestamp
        FROM {target_table}
    """)
    
    logger.info("Table Statistics:")
    stats.show()


def display_sample_data(target_table):
    """Display sample data"""
    logger.info("Sample Bronze Data:")
    
    spark.table(target_table) \
        .orderBy("_ingestion_date", "turbine_id", "timestamp") \
        .limit(10) \
        .show(truncate=False)

In [0]:
logger.info("=" * 80)
logger.info("Starting Bronze Layer Ingestion")
logger.info(f"Source: {SOURCE_PATH}")
logger.info(f"Target: {TARGET_TABLE}")
logger.info("=" * 80)

try:
    # Step 1: Read CSV
    raw_df = read_csv_files(SOURCE_PATH)
    
    # Step 2: Deduplicate
    deduped_df = deduplicate_data(raw_df)
    
    # Step 3: Merge
    merge_status = merge_into_bronze(deduped_df, TARGET_TABLE)
    logger.info(f"Merge status: {merge_status}")
    
    # Step 4: Data quality
    calculate_data_quality_metrics(TARGET_TABLE)
    
    # Step 5: Sample
    display_sample_data(TARGET_TABLE)
    
    logger.info("Bronze Layer Ingestion Complete!")
    
except Exception as e:
    logger.error(f"Failed: {str(e)}", exc_info=True)
    raise

In [0]:
spark.sql(f"DESCRIBE HISTORY {TARGET_TABLE}").show()